## gdb of population per tract for our 10 states per decade

#### Using decenial census population, downloading from https://data2.nhgis.org/review for 1950-2020 census and then 2020-2024 acs

### Downloaded here /raw/NHGIS population per tract

#### 1950 MISSING Arkansas, Iowa, Mississippi, West Virginia (census tracts were only established in selected metropolitan areas), and 1960 is missing Mississippi

In [21]:
from pathlib import Path
import pandas as pd
from zipfile import ZipFile
import geopandas as gpd
import geopandas as gpd
import pyogrio
import shutil

In [20]:

base = Path(
    "/Users/sophiameyers/Documents/GitHub/USGS-Gages-Research"
)

raw = base / "data/raw/NHGIS population per tract"
shape_folder = raw / "nhgis0002_shape"

output_folder = base / "data/processed/Population_By_State"
staging_folder = base / "data/processed/_Population_By_State_build"

states = {
    "01": ("Alabama", 26916),
    "05": ("Arkansas", 26915),
    "19": ("Iowa", 26915),
    "21": ("Kentucky", 26916),
    "28": ("Mississippi", 26915),
    "29": ("Missouri", 26915),
    "31": ("Nebraska", 26914),
    "40": ("Oklahoma", 26914),
    "47": ("Tennessee", 26916),
    "54": ("West_Virginia", 26917)
}

years = [1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020]

layer_options = {
    "TARGET_ARCGIS_VERSION": "ARCGIS_PRO_3_2_OR_LATER"
}

boundary_files = {
    1950: (
        "nhgis0002_shapefile_tl2008_us_tract_1950.zip",
        "US_tract_1950_conflated.shp"
    ),
    1960: (
        "nhgis0002_shapefile_tl2008_us_tract_1960.zip",
        "US_tract_1960_conflated.shp"
    ),
    1970: (
        "nhgis0002_shapefile_tl2008_us_tract_1970.zip",
        "US_tract_1970_conflated.shp"
    ),
    1980: (
        "nhgis0002_shapefile_tl2008_us_tract_1980.zip",
        "US_tract_1980_conflated.shp"
    ),
    1990: (
        "nhgis0002_shapefile_tl2008_us_tract_1990.zip",
        "US_tract_1990_conflated.shp"
    ),
    2000: (
        "nhgis0002_shapefile_tl2008_us_tract_2000.zip",
        "US_tract_2000_conflated.shp"
    ),
    2010: (
        "nhgis0002_shapefile_tl2010_us_tract_2010.zip",
        "US_tract_2010.shp"
    ),
    2020: (
        "nhgis0002_shapefile_tl2020_us_tract_2020.zip",
        "US_tract_2020.shp"
    )
}

output_folder.mkdir(parents=True, exist_ok=True)

existing = [
    output_folder / f"{name}_Population.gdb"
    for _, (name, _) in states.items()
]

existing = [p for p in existing if p.exists()]

if existing:
    raise FileExistsError(
        "The following geodatabases already exist. "
        "Nothing was overwritten:\n"
        + "\n".join(str(p) for p in existing)
    )

if staging_folder.exists():
    raise FileExistsError(
        f"Staging folder already exists:\n{staging_folder}\n"
        "Inspect it before continuing. Do not delete it blindly."
    )

staging_folder.mkdir(parents=True)

print("=" * 70)
print("LOADING POPULATION DATA")
print("=" * 70)

population_sources = {}

for year in years:

    if year == 1950:
        filename = "nhgis0002_ds82_1950_tract.csv"
        join_field = "GISJOIN"
        population_field = "BZ8001"

    elif year == 1960:
        filename = "nhgis0002_ds92_1960_tract.csv"
        join_field = "GISJOIN"
        population_field = "CA4001"

    else:
        filename = "nhgis0002_ts_nominal_tract.csv"
        join_field = f"GJOIN{year}"
        population_field = f"AV0AA{year}"

    if year <= 1960:
        df = pd.read_csv(
            raw / filename,
            dtype=str,
            low_memory=False
        )
    else:
        if "nominal_data" not in globals():
            nominal_data = pd.read_csv(
                raw / filename,
                dtype=str,
                low_memory=False
            )

        df = nominal_data.copy()

    required = [join_field, population_field]

    missing_columns = [
        col for col in required if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{year}: missing columns {missing_columns}"
        )

    df = df[df[population_field].notna()].copy()

    df["JOIN_KEY"] = df[join_field].str.strip()
    df["STATE_FIPS"] = df["JOIN_KEY"].str[1:3]

    df = df[
        df["STATE_FIPS"].isin(states)
    ].copy()

    numeric_population = pd.to_numeric(
        df[population_field],
        errors="coerce"
    )

    invalid = (
        df[population_field].notna()
        & numeric_population.isna()
    )

    if invalid.any():
        raise ValueError(
            f"{year}: population values could not be converted "
            f"to numbers. Inspect source data before continuing."
        )

    if (numeric_population.dropna() % 1 != 0).any():
        raise ValueError(
            f"{year}: non-integer population values detected."
        )

    df["POPULATION"] = numeric_population.astype("Int64")
    df["CENSUS_YEAR"] = pd.Series(
        year, index=df.index, dtype="int16"
    )

    # Population join keys must be unique
    if df["JOIN_KEY"].duplicated().any():
        raise ValueError(
            f"{year}: duplicate population join keys detected."
        )

    population_sources[year] = df

    print(
        f"{year}: {len(df):,} population records loaded"
    )


print("\n" + "=" * 70)
print("LOADING HISTORICAL TRACT BOUNDARIES")
print("=" * 70)

boundary_sources = {}

for year in years:

    zip_name, shp_name = boundary_files[year]
    zip_path = shape_folder / zip_name

    boundary = gpd.read_file(
        f"zip://{zip_path}!{shp_name}"
    )

    if boundary.crs is None:
        raise ValueError(
            f"{year}: source boundary has no CRS."
        )

    if "GISJOIN" not in boundary.columns:
        raise ValueError(
            f"{year}: boundary has no GISJOIN field."
        )

    boundary["GISJOIN"] = boundary["GISJOIN"].astype("string").str.strip()
    boundary["STATE_FIPS"] = boundary["GISJOIN"].str[1:3]

    boundary_sources[year] = boundary

    print(
        f"{year}: {len(boundary):,} national boundary features loaded"
    )

print("\n" + "=" * 70)
print("BUILDING STATE GEODATABASES")
print("=" * 70)

build_summary = []

for state_fips, (state_name, epsg) in states.items():

    print("\n" + "#" * 70)
    print(f"BUILDING: {state_name.upper()}")
    print(f"TARGET CRS: EPSG:{epsg}")
    print("#" * 70)

    gdb_path = (
        staging_folder /
        f"{state_name}_Population.gdb"
    )

    spatial_layers = {}
    unmatched_records = []


    for year in years:

        print(f"\nProcessing {state_name} {year}...")

        boundary = boundary_sources[year]

        state_boundary = boundary[
            boundary["STATE_FIPS"] == state_fips
        ].copy()

        pop = population_sources[year]

        state_pop = pop[
            pop["STATE_FIPS"] == state_fips
        ].copy()

        if state_pop["JOIN_KEY"].duplicated().any():
            raise ValueError(
                f"{state_name} {year}: duplicate population keys."
            )

        boundary_keys = set(
            state_boundary["GISJOIN"].dropna()
        )

        unmatched = state_pop[
            ~state_pop["JOIN_KEY"].isin(boundary_keys)
        ].copy()

        if len(unmatched):

            unmatched["AUDIT_REASON"] = (
                "Population record has no matching boundary"
            )

            unmatched_records.append(unmatched)

        pop_join = state_pop[
            ["JOIN_KEY", "POPULATION"]
        ].rename(
            columns={"JOIN_KEY": "POP_JOIN_KEY"}
        )

     
        merged = state_boundary.merge(
            pop_join,
            left_on="GISJOIN",
            right_on="POP_JOIN_KEY",
            how="left",
            validate="many_to_one"
        )

        merged["TRACT_UID"] = [
            f"{state_fips}_{year}_{i:07d}"
            for i in range(len(merged))
        ]

        merged["STATE_FIPS"] = state_fips
        merged["STATE_NAME"] = state_name
        merged["CENSUS_YEAR"] = pd.Series(
            year, index=merged.index, dtype="int16"
        )

        merged["POPULATION"] = pd.array(
            merged["POPULATION"],
            dtype="Int64"
        )

        merged = merged.set_crs(
            boundary.crs,
            allow_override=True
        )

        merged = merged.to_crs(epsg=epsg)

        layer_name = f"Tracts_{year}"

        spatial_layers[layer_name] = merged

        print(
            f"  Boundary features: {len(merged):,}"
        )

        print(
            f"  Matched population: "
            f"{merged['POPULATION'].notna().sum():,}"
        )

        print(
            f"  Unmatched population records: {len(unmatched):,}"
        )

  
    print("\nWriting feature classes...")

    first_layer = next(
        (
            name for name, gdf in spatial_layers.items()
            if len(gdf) > 0
        ),
        None
    )

    if first_layer is None:
        raise ValueError(
            f"{state_name}: no boundary features found in any year."
        )

    write_order = [
        first_layer
    ] + [
        name for name in spatial_layers
        if name != first_layer
    ]

    for i, layer_name in enumerate(write_order):

        gdf = spatial_layers[layer_name]

        if gdf.crs.to_epsg() != epsg:
            raise ValueError(
                f"{state_name} {layer_name}: incorrect CRS."
            )

    
        pyogrio.write_dataframe(
            gdf,
            gdb_path,
            layer=layer_name,
            driver="OpenFileGDB",
            append=(i > 0),
            promote_to_multi=True,
            geometry_type="MultiPolygon",
            layer_options=layer_options
        )

        print(
            f"  Written {layer_name}: {len(gdf):,} features"
        )

  
    print("Writing Population_Unmatched...")

    if unmatched_records:

        unmatched_table = pd.concat(
            unmatched_records,
            ignore_index=True,
            sort=False
        )

    else:
        # Preserve the source schema even if no unmatched
        # records exist for a state.
        unmatched_table = pd.DataFrame(
            columns=[
                "JOIN_KEY",
                "STATE_FIPS",
                "POPULATION",
                "CENSUS_YEAR",
                "AUDIT_REASON"
            ]
        )

    # Explicit integer storage
    if "POPULATION" in unmatched_table.columns:
        unmatched_table["POPULATION"] = pd.array(
            unmatched_table["POPULATION"],
            dtype="Int64"
        )

    if "CENSUS_YEAR" in unmatched_table.columns:
        unmatched_table["CENSUS_YEAR"] = pd.to_numeric(
            unmatched_table["CENSUS_YEAR"],
            errors="raise"
        ).astype("int16")

    pyogrio.write_dataframe(
        unmatched_table,
        gdb_path,
        layer="Population_Unmatched",
        driver="OpenFileGDB",
        append=True,
        layer_options=layer_options
    )

    print(
        f"  Unmatched records written: {len(unmatched_table):,}"
    )


    print("Validating geodatabase...")

    actual_layers = pyogrio.list_layers(gdb_path)
    actual_names = set(actual_layers[:, 0])

    expected_names = {
        f"Tracts_{year}" for year in years
    } | {"Population_Unmatched"}

    if actual_names != expected_names:
        raise ValueError(
            f"{state_name}: unexpected geodatabase layers.\n"
            f"Missing: {expected_names - actual_names}\n"
            f"Extra: {actual_names - expected_names}"
        )

    state_feature_count = 0

    for year in years:

        layer_name = f"Tracts_{year}"

        check = pyogrio.read_dataframe(
            gdb_path,
            layer=layer_name
        )

        expected_count = len(spatial_layers[layer_name])

        if len(check) != expected_count:
            raise ValueError(
                f"{state_name} {year}: feature count mismatch."
            )

        if check.crs.to_epsg() != epsg:
            raise ValueError(
                f"{state_name} {year}: CRS validation failed."
            )

        if "TRACT_UID" not in check.columns:
            raise ValueError(
                f"{state_name} {year}: TRACT_UID missing."
            )

        if check["TRACT_UID"].duplicated().any():
            raise ValueError(
                f"{state_name} {year}: duplicate TRACT_UID."
            )

        state_feature_count += len(check)

    unmatched_check = pyogrio.read_dataframe(
        gdb_path,
        layer="Population_Unmatched"
    )

    if len(unmatched_check) != len(unmatched_table):
        raise ValueError(
            f"{state_name}: unmatched table count mismatch."
        )

    build_summary.append({
        "STATE": state_name,
        "EPSG": epsg,
        "FEATURE_CLASSES": 8,
        "TOTAL_TRACT_FEATURES": state_feature_count,
        "UNMATCHED_POPULATION_RECORDS": len(unmatched_check),
        "VALIDATION": "PASSED"
    })

    print(
        f"\n{state_name}: VALIDATION PASSED"
    )


print("\n" + "=" * 70)
print("MOVING VALIDATED GEODATABASES TO FINAL OUTPUT")
print("=" * 70)

for _, (state_name, _) in states.items():

    source = staging_folder / f"{state_name}_Population.gdb"
    destination = output_folder / f"{state_name}_Population.gdb"

    if destination.exists():
        raise FileExistsError(
            f"Refusing to overwrite {destination}"
        )

    shutil.move(str(source), str(destination))

    print(f"  Finalized: {destination.name}")

if staging_folder.exists():
    shutil.rmtree(staging_folder)


summary = pd.DataFrame(build_summary)

summary_path = (
    base / "outputs" /
    "population_geodatabase_build_summary.csv"
)

summary_path.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(summary_path, index=False)

print("\n" + "=" * 70)
print("POPULATION GEODATABASE BUILD COMPLETE")
print("=" * 70)

print(summary.to_string(index=False))

print("Final output folder:")
print(output_folder)

print("Build summary saved:")
print(summary_path)

print("Expected geodatabases:", len(states))
print("Expected feature classes per GDB:", len(years))
print("Expected tables per GDB:", 1)
print("Total expected spatial feature classes:", len(states) * len(years))


LOADING POPULATION DATA
1950: 795 population records loaded
1960: 1,702 population records loaded
1970: 3,134 population records loaded
1980: 5,223 population records loaded
1990: 8,397 population records loaded
2000: 8,637 population records loaded
2010: 9,423 population records loaded
2020: 10,999 population records loaded

LOADING HISTORICAL TRACT BOUNDARIES
1950: 12,634 national boundary features loaded
1960: 23,096 national boundary features loaded
1970: 34,494 national boundary features loaded
1980: 46,197 national boundary features loaded
1990: 60,947 national boundary features loaded
2000: 65,312 national boundary features loaded
2010: 73,669 national boundary features loaded
2020: 85,060 national boundary features loaded

BUILDING STATE GEODATABASES

######################################################################
BUILDING: ALABAMA
TARGET CRS: EPSG:26916
######################################################################

Processing Alabama 1950...
  Boundary features

## ^^ Now we have gdb for each state with layers broken down by decade hosting population by census tracts called processed/Population_By_State

### Have some unmatched population records noted above (but compared to total amount its not extreme)